# BASTION: U.S. Airline Traffic Example

This notebook applies BASTION to **monthly U.S. international airline passenger counts (2003-2023)** and demonstrates the model on real-world data.  It is based on Section 6.1 and Appendix D of:

> Cho, J. B. & Matteson, D. S. (2026). *BASTION: A Bayesian Framework for Trend and Seasonality Decomposition.* arXiv:2601.18052.

## Why this dataset?

The airline traffic series has a textbook structure:
- A **long-term upward trend** in international travel.
- A pronounced **annual seasonality** (peak in July/August, trough in Jan/Feb).
- A sharp **abrupt structural break** in early 2020 caused by COVID-19 travel restrictions.

Capturing this structural break while producing a smooth, interpretable trend is a challenge for most decomposition methods.  BASTION's horseshoe prior on second differences allows the trend to adapt sharply where needed without introducing noise elsewhere.  As the paper notes (Section 6.1):

> *'BASTION provides a smooth yet adaptive trend, capturing the abrupt COVID-19 drop while preserving overall structure.'*

## What this notebook covers

1. **Load and explore** the dataset.
2. **Fit BASTION** with an annual seasonal component (`K = 12`).
3. **Visualise the signal** (trend + seasonality) with 95% credible bands.
4. **Full decomposition** -- trend, seasonality, and remainder.
5. **Metrics table** from the paper's simulation study placing results in context.

In [ ]:
import os, sys, warnings
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
PROJECT_DIR  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
OUTPUT_DIR   = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

sys.path.insert(0, PROJECT_DIR)
from pybastion import fit_BASTION
from pybastion.datasets import load_airtraffic as _load_raw

print(f'pybastion loaded from: {PROJECT_DIR}')

## 1. Load and Explore the Data

Monthly passenger counts (millions) for U.S. international routes, sourced from Kaggle (Yan, 2023).  The dataset is bundled with `pybastion` via `load_airtraffic()`.

In [ ]:
def load_airtraffic():
    df = _load_raw()
    df['date'] = pd.to_datetime(
        df['Year'].astype(int).astype(str) + '-'
        + df['Month'].astype(int).astype(str).str.zfill(2) + '-01')
    return df.rename(columns={'Int_Pax': 'pax'})[['date', 'pax']].reset_index(drop=True)

air = load_airtraffic()
print(f'Observations : {len(air)} monthly records')
print(f'Date range   : {air["date"].min().date()} to {air["date"].max().date()}')
print(f'Pax range    : {air["pax"].min():,.0f} to {air["pax"].max():,.0f}')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(air['date'], air['pax'], linewidth=1.0, color='black')
ax.axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--',
           linewidth=1.2, label='COVID-19 (Mar 2020)')
ax.set_xlabel('Date'); ax.set_ylabel('Passengers (M)')
ax.set_title('U.S. Monthly International Airline Traffic 2003-2023')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 2. Fit BASTION

We fit a single annual seasonal component (`K = 12`).  Outlier detection is enabled -- the horseshoe+ prior will shrink most months to zero while flagging genuinely anomalous points (e.g. the COVID-19 shock).

**MCMC settings:**

| Parameter | Value | Note |
|-----------|-------|------|
| `Ks` | `[12]` | Annual seasonality (monthly data) |
| `Outlier` | `True` | Horseshoe+ outlier component |
| `nsave` | 2 000 | Saved draws per chain |
| `nburn` | 5 000 | Burn-in |
| `nskip` | 4 | Thinning interval |
| `nchains` | 2 | Sequential independent chains |

Total steps per chain: **15 000**.  Expected runtime: ~30-60 min.

In [ ]:
result  = fit_BASTION(
    air['pax'].values,
    Ks=[12],
    Outlier=True,
    cl=0.95,
    obsSV='const',
    nsave=2000,
    nburn=5000,
    nskip=4,
    nchains=2,
    seed=40,
)
summary = result['summary']
print('Estimated components:', [k for k in summary if k.endswith('_sum')])

## 3. Signal: Trend + Seasonality

The plot below shows the 2016-2023 period, where the COVID-19 structural break is visible.  Black line = posterior mean signal; red = trend alone; grey = 95% credible region.  Notice how BASTION captures the sharp 2020 drop in the **trend** while the seasonal component remains well-structured.

In [ ]:
mask        = air['date'] >= '2016-01-01'
dates_sub   = air['date'].values[mask]
pax_sub     = air['pax'].values[mask]
idx         = np.where(mask)[0]

trend_mean  = np.asarray(summary['Trend_sum']['Mean']).ravel()[idx]
trend_lo    = np.asarray(summary['Trend_sum']['CR_lower']).ravel()[idx]
trend_hi    = np.asarray(summary['Trend_sum']['CR_upper']).ravel()[idx]
signal_mean = np.asarray(summary['Signal_sum']['Mean']).ravel()[idx]
signal_lo   = np.asarray(summary['Signal_sum']['CR_lower']).ravel()[idx]
signal_hi   = np.asarray(summary['Signal_sum']['CR_upper']).ravel()[idx]

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(dates_sub, pax_sub, s=10, color='black', zorder=1, label='Observed')
ax.fill_between(dates_sub, signal_lo, signal_hi, color='grey', alpha=0.35, zorder=2,
                label='95% CI')
ax.plot(dates_sub, signal_mean, linewidth=1.5, color='black', zorder=3,
        label='Signal (Trend + Season)')
ax.plot(dates_sub, trend_mean,  linewidth=1.5, color='red',   zorder=4,
        label='Trend')
ax.set_xlabel('Date'); ax.set_ylabel('Passengers (M)')
ax.set_title('BASTION: signal estimate -- U.S. airline traffic 2016-2023')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'airtraffic_signal.pdf'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Full Decomposition

The four panels show the complete BASTION decomposition across the 2003-2023 series:

- **Observed data** with signal overlaid.
- **Trend** -- smooth long-run level with 95% credible band.
- **Seasonality** (period 12) -- repeating annual cycle with 95% credible band.
- **Remainder** -- residuals after removing trend and seasonality.

In [ ]:
dates = air['date'].values
pax   = air['pax'].values

trend_m_all = np.asarray(summary['Trend_sum']['Mean']).ravel()
trend_lo_all = np.asarray(summary['Trend_sum']['CR_lower']).ravel()
trend_hi_all = np.asarray(summary['Trend_sum']['CR_upper']).ravel()
signal_all   = np.asarray(summary['Signal_sum']['Mean']).ravel()
s12_mean     = np.asarray(summary['Seasonal12_sum']['Mean']).ravel()
s12_lo       = np.asarray(summary['Seasonal12_sum']['CR_lower']).ravel()
s12_hi       = np.asarray(summary['Seasonal12_sum']['CR_upper']).ravel()
remainder    = pax - signal_all

fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

ax = axes[0]
ax.scatter(dates, pax, s=6, color='black', zorder=2, alpha=0.7)
ax.plot(dates, signal_all,  linewidth=1.4, color='black', zorder=3, label='Signal')
ax.plot(dates, trend_m_all, linewidth=1.4, color='red',   zorder=4, label='Trend')
ax.set_title('Observed Data'); ax.set_ylabel('Passengers (M)'); ax.legend(fontsize=8)

ax = axes[1]
ax.fill_between(dates, trend_lo_all, trend_hi_all, color='grey', alpha=0.4)
ax.plot(dates, trend_m_all, linewidth=1.4, color='black')
ax.set_title('Trend'); ax.set_ylabel('Passengers (M)')

ax = axes[2]
ax.fill_between(dates, s12_lo, s12_hi, color='grey', alpha=0.4)
ax.plot(dates, s12_mean, linewidth=1.4, color='black')
ax.set_title('Seasonality (period 12)'); ax.set_ylabel('Passengers (M)')

ax = axes[3]
ax.plot(dates, remainder, linewidth=0.9, color='black')
ax.axhline(0, color='grey', linestyle='--', linewidth=0.5)
ax.set_title('Remainder'); ax.set_ylabel('Passengers (M)')

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'airtraffic_decomposition.pdf'), dpi=150,
            bbox_inches='tight')
plt.show()

## 5. Simulation-Average Metrics (Cho & Matteson 2026, Tables 2 & 3)

The tables below show results for **DGP 4** -- the simulation scenario closest to this application: piecewise-linear trend, two seasonal components, stochastic volatility, and additive outliers.  Competitor methods are shown for reference only; they cannot be run via `pybastion`.

### Mean Squared Error (lower is better)

| Method | Signal MSE | Trend MSE | Seasonality MSE |
|--------|------------|-----------|----------------|
| TBATS  | 11.829 | 11.111 | 5.364 |
| MSTL   | 11.430 | 12.328 | 3.045 |
| STR    | 20.548 | 13.431 | 11.358 |
| **BASTION** | **2.877** | **5.210** | **2.562** |

### Empirical Coverage at 95% (higher is better)

| Component | STR coverage | BASTION coverage | STR width | BASTION width |
|---|---|---|---|---|
| Signal | 0.679 | **0.981** | 5.148 | 5.606 |
| Trend | 0.668 | **0.939** | 2.296 | 2.246 |
| Seasonality | 0.623 | **0.999** | 2.852 | 3.360 |

*Source: Tables 2 & 3, DGP 4 in Cho & Matteson (2026).*

DGP 4 is where BASTION's joint modelling of outliers and stochastic volatility makes the biggest difference -- STR's coverage collapses to 0.68 for the signal while BASTION stays above 0.98.